# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library. Example analyses are demonstrated for record sets and fields using their Croissant schema `@id`s, to ensure robust, reusable access.

### Dataset Source
The dataset is described by a Croissant schema JSON-LD file at:  
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed (run cell if `mlcroissant` not available)
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. The dataset metadata gives high-level information, available record sets, and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata (do NOT subscript the object, access members as attributes)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their IDs (`@id`s), and examine their fields and columns. This step will show all high-level data structures before extracting any records.

In [ ]:
# List all record sets and their field IDs using Croissant schema
if hasattr(metadata, 'record_sets'):
    all_record_sets = metadata.record_sets
elif hasattr(metadata, 'record_set'):
    # Croissant sometimes singular/plural
    all_record_sets = metadata.record_set
else:
    all_record_sets = []

if not all_record_sets:
    # If no recordSet attribute on the metadata, attempt to load all file objects (Croissant 1.0)
    all_record_sets = []
    if hasattr(metadata, 'distributions'):
        for dist in metadata.distributions:
            if hasattr(dist, 'record_sets'):
                all_record_sets.extend(dist.record_sets)

print('Record sets and their field/column IDs:')
record_set_ids = []
for rs in dataset.record_sets:
    rs_meta = dataset.record_set(rs)
    print(f'Record set @id: {rs}')
    if hasattr(rs_meta, 'fields'):
        field_ids = [field['@id'] if isinstance(field,dict) else field for field in rs_meta.fields]
        print(f'  Fields: {field_ids}')
    if hasattr(rs_meta, 'columns'):
        column_ids = [col['@id'] if isinstance(col,dict) else col for col in rs_meta.columns]
        print(f'  Columns: {column_ids}')
    record_set_ids.append(rs)

In [ ]:
# Optionally, print a sample record from each record set
for rs_id in record_set_ids:
    try:
        sample = next(dataset.records(record_set=rs_id))
        print(f'Sample record from record set {rs_id}:')
        for k in sample:
            print(f'  {k}: {sample[k]}')
    except Exception as e:
        print(f'Could not get sample from {rs_id}: {e}')

## 3. Data Extraction
Extract all records for each record set by its `@id`, and load them into Pandas DataFrames for further inspection and analysis.

In [ ]:
# Load all record sets into DataFrames, keyed by record set @id
dataframes = {}
for rs_id in record_set_ids:
    try:
        recs = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f'Loaded {df.shape[0]} records for {rs_id}, columns: {df.columns.tolist()}')
    except Exception as e:
        print(f'Could not load records for {rs_id}: {e}')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering by a numeric field, normalization, and grouping/categorization, using column names corresponding to field or column `@id`s as shown in the previous step.

In [ ]:
# Choose a record set with numeric fields for demonstration:
chosen_rs_id = None
for rs_id, df in dataframes.items():
    # Find a DataFrame with at least one float/int column
    numeric_cols = df.select_dtypes(include=['number']).columns
    if len(numeric_cols) > 0:
        chosen_rs_id = rs_id
        break
if chosen_rs_id is None:
    print('No record set with numeric fields found!')
else:
    df = dataframes[chosen_rs_id]
    print(f'Using record set: {chosen_rs_id} for EDA')
    print('Numeric columns:', df.select_dtypes(include=['number']).columns.tolist())

In [ ]:
# Now filter, normalize and group by a categorical field if one is available
if chosen_rs_id is not None:
    df = dataframes[chosen_rs_id]
    num_cols = df.select_dtypes(include=['number']).columns
    # Use the first numeric column
    if len(num_cols) > 0:
        numeric_field_id = num_cols[0]
        print(f"Applying filter to column {numeric_field_id}")
        
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)}")
        
        # Normalize column
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print("First 5 normalized records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Group by first non-numeric field
        group_col = None
        for c in df.columns:
            if c not in num_cols:
                group_col = c
                break
        if group_col:
            grouped = filtered_df.groupby(group_col)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} grouped by {group_col}:")
            print(grouped.head())
        else:
            print("No categorical column found for grouping.")

## 5. Visualization
Visualize data distributions and relationships. We use matplotlib/seaborn and show, for example, the histogram of the chosen numeric field, and if applicable, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs_id is not None and len(num_cols) > 0:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    # Boxplot by group if group_col exists
    if group_col:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_col, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_col}')
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
This notebook provided a step-by-step approach to exploring the FAIR^2 dataset using `mlcroissant`, referencing all record sets, fields, and columns by their Croissant `@id`. You can adapt the notebook for in-depth analysis, further feature engineering, or use the DataFrames for your own models and insights.